# 04 · Federated transfer: elastic-net Cox, XGBoost and predictive checklist

Wrapper notebook for the second federated delivery (`rhino_data_v2`). All scoring logic lives in
`survival_common/federated_endpoint_transfer.py` and `federated_models/federated_clinical_comparison.py`. It runs
only the analyses the wired manuscript figures (`COMPASS/figures/retained_exports.txt`: Figures 5 and 6) read.

Three platinum-trained model families, each refit **without Dana-Farber** (Fred Hutch, JHU and MSK), are scored on the
full local DFCI ADT cohort. No DFCI patient was in their training data, so this is an external validation on all three
endpoints: **time to NEPC**, **time to AVPC** and **time to platinum** (the training endpoint, as a same-endpoint
reference).

| Family | v2 location (no-DFCI refit) | Scored as |
|---|---|---|
| Elastic-net Cox | `federated_elastic_net_cox_no_DFCI/` (landmarks 0/90/180 in one bundle) | linear predictor |
| XGBoost | `federated_xgboost*/lm_{0,90,180}/no_DFCI/` (one bundle per landmark); if absent, the first delivery's `federated_xgboost_no_DFCI/` | boosted margin |
| Predictive checklist | `feature_set_models/no_DFCI/predictive_checklists/` (adjust-only refit; 365- and 1825-day models) | raw item count, plus frozen probability and flag |

Dynamic DeepHit and the regular (non-feature-set) checklist are not scored here: no wired figure reads them.

Bundles are found by file name anywhere under `RHINO_ROOT` in a `no_DFCI` / `*_no_DFCI` directory, so the
exact XGBoost folder name do not matter. `feature_set_models/` is skipped by that search: the
predictive checklist family is read from its fixed path there, and the regular checklist is scored separately (last cell), never merged with it. Each family's per-landmark bundles are merged into one
(`merge_bundles`): the shared fields (format version, analysis label, time unit, library version) must agree, and
a landmark exported twice is an error. The pooled preprocessing in each bundle is applied as shipped and never refit.

A family in `V1_FALLBACK_FAMILIES` (XGBoost by default) with no no-DFCI bundle in `rhino_data_v2` is scored from the
first delivery instead (`rhino_results`, overridden by `COMPASS_RHINO_V1_ROOT`). The bundle summary and
`transfer_run.json` record which delivery each family came from. Both deliveries were refit on Fred Hutch, JHU
and MSK, so neither has seen a DFCI patient, but they are separate federated runs.

**Metrics** for each endpoint × landmark × model: mean AUC(t) with a patient-bootstrap 95% CI (the
figure metric), graded as in the federated runs: a week grid rebuilt on each endpoint × landmark cohort with the
federated rule (25th–75th percentile event weeks, at most 25 points), the same support filter and coverage gates
(≥ 2 valid horizons covering ≥ 50% of the grid, else NaN), censoring weights from the local cohort, paired ΔAUC(t) and ΔC vs the age-only `baseline`,
Harrell's and Uno's C, per-horizon AUC(t), and Cox HR per SD of the score.
- The landmark XGBoost packages export no age-only model, so XGBoost is paired with the elastic-net Cox
  `baseline` (`baseline_bundle` in the metrics). A first-delivery XGBoost bundle with its own age-only model keeps
  it. An age-only model's discrimination depends only on how it ranks age,
  so this is the same comparator.
- Checklists carry one row per model horizon, scored at that horizon.

**Outputs** (all under `OUTPUT_DIR` = `rhino_data_v2/federated_transfer_adt_no_DFCI`)
- `transfer_metrics.csv`, `transfer_auc_t.csv`: elastic-net Cox and XGBoost only, the federated Figure 5 inputs.
- `transfer_clinical_comparison.csv` (plus `transfer_clinical_cox_coefs.csv`, `transfer_subpop_risk_summary.csv`) and
  `transfer_km_patients.csv`: Cox and XGBoost only. `transfer_km_patients.csv` feeds the Figure 5 KM panels; the
  clinical rows feed the `__no_d` and mean-landmark alternates.
- `transfer_checklist_patients.csv` and `transfer_checklist_comparison.csv`: the no-DFCI 1-year feature-set platinum
  checklist from +180 days (federated Figure 6). The patient table gives the panel c–e groups on the ADT clock. The
  comparison table gives panel b: the checklist, Gleason ≥8 and any TP53/RB1/PTEN alteration as yes/no rules in the
  same sequenced patients with a Gleason score, on the sequencing clock.
- `checklist_grid/`: the same two Figure 6 tables at every landmark (0/90/180) × horizon (1/5 years), for the
  per-cell checklist panels.
- `checklist_grid_guidelines/`: the same grid for the guideline checklist (thresholds rounded to practical values), with
  `transfer_checklist_guideline_items.csv` holding the rounded and exact thresholds.

After this notebook, rerun `05_prep_figure_data.ipynb`, then `COMPASS/figures/run_figures.R`, which read `rhino_data_v2` by default.

**Transferred model vs Gleason and TP53/RB1/PTEN** (the federated Figure 5 alternates):
- **b · `gleason`**: patients with a Gleason score (ADT clock). Local Cox: **Gleason (continuous) + age**.
- **c · `trio`**: sequenced patients (sequencing clock). Local Cox: **TP53_ANY + RB1_ANY + PTEN_ANY + age**.
- **d · `trio_gleason`**: sequenced patients with a Gleason score (sequencing clock). Local Cox:
  **Gleason + TP53_ANY + RB1_ANY + PTEN_ANY + age**.

Each compares the transferred `both` score, the 5-fold out-of-fold local Cox, and a combined local Cox of the
transferred score + the clinical covariates without age. The somatic arms are anchored on the sequencing report date
(closest to ADT start); no somatic data is used on the ADT clock. Clinical data must be available strictly before the
landmark.

## Configuration

In [ ]:
import os, re, sys, json
from pathlib import Path

sys.path.insert(0, "../pipeline")
import compass_pipeline as cp

PROJECT_ROOT = cp.PROJECT_ROOT
for _p in (str(PROJECT_ROOT), str(cp.COMPASS_DIR / "federated_models")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import numpy as np
import pandas as pd

from survival_common.federated_inference import load_bundle, merge_bundles
from survival_common import federated_endpoint_transfer as fet
from survival_common import federated_scoring as fs

DATA_ROOT = cp._PROFILE_OUTPUT_ROOT

# >>> Edit: second federated delivery; transfer uses the bundles refit without DFCI (Fred Hutch + JHU + MSK) <<<
RHINO_ROOT = Path(os.environ.get("COMPASS_RHINO_ROOT", str(Path(DATA_ROOT) / "rhino_data_v2")))
# First delivery: a family listed here with no no-DFCI bundle in RHINO_ROOT is scored from this one instead.
RHINO_V1_ROOT        = Path(os.environ.get("COMPASS_RHINO_V1_ROOT", str(Path(DATA_ROOT) / "rhino_results")))
V1_FALLBACK_FAMILIES = ("xgboost_cox",)   # () = rhino_data_v2 only

DATA_PATH      = Path(DATA_ROOT) / "longitudinal_prediction_data_adt.csv"
OUTPUT_DIR     = RHINO_ROOT / "federated_transfer_adt_no_DFCI"
ALL_MODELS_DIR = OUTPUT_DIR / "all_models"
FEATURE_SET_DIR = "feature_set_models"   # adjust-only refits; the default predictive checklist comes from here
LAB_MAPPING    = fet.DEFAULT_LAB_MAPPING

ENDPOINTS   = ("nepc", "avpc", "platinum")
EXCLUSION   = "none"            # "none" = full ADT cohort; "pre_anchor_castrate" for noprecastrate bundles
LANDMARKS   = None              # None = every landmark present in the bundles
HORIZONS    = fet.DEFAULT_HORIZONS_DAYS   # checklist model horizons (days); other families use the federated week grid
N_BOOTSTRAP = 200
SEED        = 0

# Figure 5 b-d: transferred labs score vs local Gleason / TP53-RB1-PTEN Cox
import federated_clinical_comparison as fcc
GLEASON_PATH          = fcc.sg.DEFAULT_GLEASON_PATH
SOMATIC_PATH          = fcc.sg.DEFAULT_SOMATIC_PATH
SOMATIC_MANIFEST_PATH = fcc.sg.DEFAULT_SOMATIC_MANIFEST_PATH
CLINICAL_CV_FOLDS     = 5     # out-of-fold folds for the local Cox
CLINICAL_MIN_EVENTS   = 5     # fewer events in a subpopulation -> count-only row

# Bundle and export-metrics file per family. The landmark packages write the same file name once per
# landmark (lm_0 / lm_90 / lm_180); the elastic-net Cox bundle holds every landmark.
BUNDLE_FILES = {
    "elastic_net_cox":      "cox_federated_elasticnet_model_adt.json",
    "xgboost_cox":          "xgboost_federated_model_adt.json",
    "predictive_checklist": "predictive_checklist_federated_model_adt.json",
}
METRIC_FILES = {
    "elastic_net_cox":      "cox_federated_elasticnet_metrics_adt.csv",
    "xgboost_cox":          "xgboost_federated_metrics_adt.csv",
    "predictive_checklist": "predictive_checklist_metrics_adt.csv",
}
PRIMARY_FAMILIES = ("elastic_net_cox", "xgboost_cox")   # Figure 5 and the clinical comparison
BASELINE_DONOR   = {"xgboost_cox": "elastic_net_cox"}    # landmark XGBoost exports no age-only model
                                                         # (borrowed only by a bundle without its own)

# The default predictive checklist is the feature-set refit; the regular one is scored by the last cell.
FEATURE_SET_BUNDLE = (RHINO_ROOT / FEATURE_SET_DIR / "no_DFCI" / "predictive_checklists"
                      / BUNDLE_FILES["predictive_checklist"])

def is_no_dfci(path, root):
    return any(part == "no_DFCI" or part.endswith("_no_DFCI") for part in path.relative_to(root).parts)

def natural_key(path):   # lm_90 before lm_180
    return [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", str(path))]

def find_bundles(root, name):   # skips each delivery's own transfer output and the feature-set refits
    skip = {OUTPUT_DIR.name, FEATURE_SET_DIR}
    return sorted((p for p in root.rglob(name)
                   if is_no_dfci(p, root) and not skip & set(p.relative_to(root).parts)), key=natural_key)

BUNDLE_PATHS, DELIVERY = {}, {}   # family -> bundle paths, and the delivery they came from
for family, name in BUNDLE_FILES.items():
    if family == "predictive_checklist":
        BUNDLE_PATHS[family], DELIVERY[family] = [FEATURE_SET_BUNDLE] if FEATURE_SET_BUNDLE.exists() else [], RHINO_ROOT
        continue
    BUNDLE_PATHS[family], DELIVERY[family] = find_bundles(RHINO_ROOT, name), RHINO_ROOT
    if not BUNDLE_PATHS[family] and family in V1_FALLBACK_FAMILIES and RHINO_V1_ROOT.is_dir():
        BUNDLE_PATHS[family], DELIVERY[family] = find_bundles(RHINO_V1_ROOT, name), RHINO_V1_ROOT
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for family, paths in BUNDLE_PATHS.items():
    print(f"{family}: {len(paths)} no-DFCI bundle(s) in {DELIVERY[family]}")
    for p in paths:
        print("  ", p.relative_to(DELIVERY[family]))
    if paths and DELIVERY[family] != RHINO_ROOT:
        print(f"   WARNING: none in {RHINO_ROOT.name}; scoring the {DELIVERY[family].name} bundle instead")

## Load and merge bundles

One bundle per family, keyed by its file stem. A family with no `rhino_data_v2` delivery is scored from the first
delivery when it is in `V1_FALLBACK_FAMILIES` (XGBoost by default) and skipped with a warning otherwise; a landmark
delivered twice, or deliveries that disagree on format, label, time unit or library version, stop here.

In [ ]:
def load_family(family, paths):
    """Load one family's per-landmark bundles and merge them into one."""
    parts = [load_bundle(p) for p in paths]
    for p, b in zip(paths, parts):
        assert b.get("model_family") == family, f"{p}: model_family {b.get('model_family')!r}, expected {family!r}"
    return merge_bundles(parts, sources=[str(p) for p in paths])

bundles, SOURCES, rows = {}, {}, []
for family, paths in BUNDLE_PATHS.items():
    if not paths:
        where = f"{RHINO_ROOT} or {RHINO_V1_ROOT}" if family in V1_FALLBACK_FAMILIES else RHINO_ROOT
        print(f"WARNING: no no-DFCI {family} bundle ({BUNDLE_FILES[family]}) under {where}; skipped")
        continue
    stem = Path(BUNDLE_FILES[family]).stem
    b = bundles[stem] = load_family(family, paths)
    SOURCES[stem] = paths
    label = str(b.get("analysis_label") or "")
    if ("noprecastrate" in label) != (EXCLUSION == "pre_anchor_castrate"):
        print(f"WARNING: {stem} analysis_label={label!r} but EXCLUSION={EXCLUSION!r}")
    for m in b["models"]:
        rows.append({
            "bundle": stem, "model_family": family, "analysis_label": label,
            "trained_endpoint": b.get("endpoint", "platinum"),
            "landmark_days": m["landmark_days"], "config": m.get("config"),
            "training_arm": m.get("training_arm"), "model_horizon_days": m.get("horizon_days"),
        })

STEMS = {b["model_family"]: stem for stem, b in bundles.items()}
PRIMARY = [STEMS[f] for f in PRIMARY_FAMILIES if f in STEMS]
assert PRIMARY, f"no elastic-net Cox or XGBoost no-DFCI bundle under {RHINO_ROOT}"
# A first-delivery XGBoost bundle carries its own age-only model and keeps it.
BASELINE_FALLBACK = {STEMS[f]: STEMS[d] for f, d in BASELINE_DONOR.items() if f in STEMS and d in STEMS
                     and not any(m.get("config") == "baseline" for m in bundles[STEMS[f]]["models"])}
print("age-only baseline borrowed:", BASELINE_FALLBACK or "none")

def _values(s):
    return ", ".join(str(v) for v in sorted(s.dropna().unique())) or "–"

bundle_summary = pd.DataFrame(rows)
(bundle_summary.groupby(["model_family", "bundle"], sort=False)
    .agg(n_models=("config", "size"), landmarks=("landmark_days", _values), configs=("config", _values),
         arms=("training_arm", _values), model_horizons=("model_horizon_days", _values))
    .assign(delivery=lambda d: [DELIVERY[f].name for f in d.index.get_level_values("model_family")],
            sources=lambda d: [", ".join(str(p.parent.relative_to(DELIVERY[f])) for p in SOURCES[s])
                               for f, s in d.index]))

## Local cohort

In [ ]:
local = fet.read_local_frame(DATA_PATH, ENDPOINTS)
lab_mapping = fet.load_lab_mapping(LAB_MAPPING)

pt = local.drop_duplicates(fet.LOCAL_ID_COL)
print(f"{len(local):,} rows, {len(pt):,} patients from {DATA_PATH}")
pd.DataFrame({
    ep: {"events": int(pt[ind].fillna(0).astype(int).sum()),
         "dated events": int((pt[ind].fillna(0).astype(int).eq(1) & pt[dt].notna()).sum())}
    for ep, (ind, dt) in fet.ENDPOINT_SOURCES.items() if ep in ENDPOINTS
}).T

## Run transfer evaluation

The Figure 5 bundles (elastic-net Cox, XGBoost) are scored and written first. The bootstrap resamples depend only
on the seed and the (endpoint, landmark) frame, never on which bundles are scored, so the checklist run
below shares them.

In [ ]:
# Shared by every run so their bootstrap resamples and horizons match.
TRANSFER_KW = dict(
    endpoints=ENDPOINTS, lab_mapping=lab_mapping, exclusion=EXCLUSION,
    landmarks=LANDMARKS, horizons_days=HORIZONS,
    n_bootstrap=N_BOOTSTRAP, seed=SEED,
)

def combine(*results):
    """Row-bind transfer results scored on the same frames; prep_stats is per endpoint, so it is the first's."""
    return {k: pd.concat([r[k] for r in results], ignore_index=True)
               if isinstance(results[0][k], pd.DataFrame) else results[0][k]
            for k in results[0]}

def write_outputs(res, stems, out_dir, stem_bundles=None):
    stem_bundles = bundles if stem_bundles is None else stem_bundles
    out_dir.mkdir(parents=True, exist_ok=True)
    for name in ("metrics", "auc_t"):
        if not res[name].empty:
            res[name].to_csv(out_dir / f"transfer_{name}.csv", index=False)
    with open(out_dir / "transfer_run.json", "w") as fh:
        json.dump({
            "data": str(DATA_PATH), "rhino_root": str(RHINO_ROOT),
            "bundles": {s: {"paths": [str(p) for p in SOURCES[s]],
                            "delivery": str(DELIVERY[stem_bundles[s]["model_family"]]),
                            **{k: stem_bundles[s].get(k) for k in (
                                "model_family", "analysis_label", "format_version",
                                "xgboost_version", "torch_version", "training_sites")}}
                        for s in stems},
            "baseline_fallback": {k: v for k, v in BASELINE_FALLBACK.items() if k in stems},
            "endpoints": list(ENDPOINTS), "exclusion": EXCLUSION,
            "landmarks": list(LANDMARKS) if LANDMARKS else None,
            "horizons_days": list(HORIZONS), "n_bootstrap": N_BOOTSTRAP, "seed": SEED,
            "prep_stats": res["prep_stats"],
        }, fh, indent=2, default=str)
    print(f"wrote outputs -> {out_dir}")

primary_bundles = {s: bundles[s] for s in PRIMARY}
res = fet.evaluate_transfer(local, primary_bundles, baseline_fallback=BASELINE_FALLBACK, **TRANSFER_KW)
write_outputs(res, PRIMARY, OUTPUT_DIR)   # Figure 5: one row per endpoint × bundle × landmark × config

In [ ]:
# Checklists (each paired with its own age-only model) share the Figure 5 frames and bootstrap resamples; their
# scores feed the Figure 6 tables below. Their metrics are not written: no wired figure reads them.
EXTRA = [s for s in bundles if s not in PRIMARY]
if EXTRA:
    extra_res = fet.evaluate_transfer(local, {s: bundles[s] for s in EXTRA}, **TRANSFER_KW)
    all_res = combine(res, extra_res)
else:
    print("no checklist bundle; Figure 6 inputs are not written")
    all_res = res


In [ ]:
# Trio arm (Figure 5 c) on the sequencing clock: the Cox/XGBoost bundles re-scored with each sequenced patient's
# sequencing report date (closest to ADT start) as the anchor, so labs come from before sequencing + landmark and
# outcomes run from sequencing + landmark.
clin_gleason, clin_somatic, clin_somatic_features = fcc.load_clinical_sources(
    GLEASON_PATH, SOMATIC_PATH, SOMATIC_MANIFEST_PATH)
specimen = fcc.select_specimen(local, clin_somatic, clin_somatic_features)
local_seq = local
if EXCLUSION != "none":
    # Judge the exclusion on the ADT clock, then re-anchor the patients it keeps.
    kept, _ = fs.apply_exclusion(fet.to_federated_long(local, "platinum", lab_mapping)[0], EXCLUSION)
    local_seq = local[pd.to_numeric(local[fet.LOCAL_ID_COL], errors="coerce").isin(kept[fs.ID_COL].unique())]
local_seq = fcc.sequencing_anchored_local(local_seq, specimen)
seq_offset = specimen[fcc.sg.SEQUENCING_TO_ADT_DAYS]
print(f"{local_seq[fet.LOCAL_ID_COL].nunique():,} sequenced patients; sequencing − ADT start median "
      f"{seq_offset.median():.0f}d (IQR {seq_offset.quantile(.25):.0f} to {seq_offset.quantile(.75):.0f}d), "
      f"{int(seq_offset.lt(0).sum()):,} sequenced before ADT")

res_seq = fet.evaluate_transfer(local_seq, primary_bundles, baseline_fallback=BASELINE_FALLBACK,
                                **{**TRANSFER_KW, "exclusion": "none"})

In [ ]:
# Figure 6 b: the frozen checklists re-scored on the sequencing clock, so the flag is compared with Gleason and
# TP53/RB1/PTEN status on the same sequenced patients.
CHECKLIST_STEMS = [s for s in bundles if bundles[s]["model_family"] == "predictive_checklist"]
if CHECKLIST_STEMS:
    res_seq_checklist = fet.evaluate_transfer(local_seq, {s: bundles[s] for s in CHECKLIST_STEMS},
                                              **{**TRANSFER_KW, "exclusion": "none"})
else:
    print("no checklist bundle; Figure 6 inputs are not written")

## Transferred model vs Gleason and TP53/RB1/PTEN, and Figure 6 checklist tables

Inputs for federated Figures 5 (clinical comparison, KM patients) and 6 (checklist patients, rule comparison, grid).
The scores are the frozen bundles, never refit; `res` is the ADT clock and `res_seq` the sequencing clock.


In [ ]:
lms = sorted(res["scores"]["landmark_days"].unique())
# Gleason arm on the ADT clock (Gleason and age only: no somatic data on the ADT clock); trio and
# trio + Gleason arms on the sequencing clock (trio taken from the anchoring specimen, Gleason available
# before sequencing + landmark).
clinical = fcc.build_clinical_features(local, lms, gleason=clin_gleason)
clinical_seq = fcc.build_clinical_features(
    local_seq, lms, gleason=clin_gleason, specimen=specimen, somatic_features=clin_somatic_features)


In [ ]:
def subpop_summary(scores_adt, scores_seq):
    """Score strata: full cohort and Gleason on the ADT clock, sequencing strata on the sequencing clock."""
    return pd.concat([
        fcc.subpopulation_score_summary(scores_adt, clinical, sections=("cohort", "gleason"))
            .assign(time_origin="ADT start"),
        fcc.subpopulation_score_summary(scores_seq, clinical_seq, sections=("sequencing",))
            .assign(time_origin="sequencing date"),
    ], ignore_index=True)

comp, coefs = fcc.compare_by_subpopulation(
    {"gleason": (res["scores"], clinical), "trio": (res_seq["scores"], clinical_seq),
     "trio_gleason": (res_seq["scores"], clinical_seq)},
    n_folds=CLINICAL_CV_FOLDS, n_bootstrap=N_BOOTSTRAP,
    seed=SEED, min_events=CLINICAL_MIN_EVENTS)
subpop = subpop_summary(res["scores"], res_seq["scores"])
# Patient-level labs scores + strata for the Figure 5 e–h KM panels (score groups are assigned in R).
km = fcc.km_patients({"adt": (res["scores"], clinical), "sequencing": (res_seq["scores"], clinical_seq)})
# Figure 6: checklist flags on the ADT clock (c–e); flag vs Gleason ≥8 vs any TP53/RB1/PTEN alteration on the same
# sequenced patients (b). Written only when a checklist bundle was scored.
checklist_patients = checklist_comparison = checklist_full = None
if CHECKLIST_STEMS:
    checklist_patients = fcc.checklist_patients(all_res["scores"], landmark=180, horizon=365)
    checklist_comparison = fcc.checklist_rule_comparison(res_seq_checklist["scores"], clinical_seq,
                                                         n_bootstrap=N_BOOTSTRAP, seed=SEED,
                                                         landmark=180, horizon=365)
    # Panel b: the checklist's accuracy per endpoint in the full within-DFCI ADT cohort (no Gleason/sequencing filter).
    checklist_full = fcc.checklist_full_accuracy(all_res["scores"], n_bootstrap=N_BOOTSTRAP, seed=SEED,
                                                 landmark=180, horizon=365)
fcc.write_outputs(comp, coefs, subpop, OUTPUT_DIR, km=km, checklist=checklist_patients,
                  checklist_comparison=checklist_comparison, checklist_full_accuracy=checklist_full)
if CHECKLIST_STEMS:
    # Supplement: Figure 6 at every landmark × horizon (federated_checklist_transfer).
    fcc.write_checklist_grid(*fcc.checklist_grid(all_res["scores"], res_seq_checklist["scores"], clinical_seq,
                                                 n_bootstrap=N_BOOTSTRAP, seed=SEED), OUTPUT_DIR,
                            full_accuracy=fcc.checklist_full_accuracy_grid(all_res["scores"], n_bootstrap=N_BOOTSTRAP,
                                                                           seed=SEED))


## Guideline checklists (generalized federated checklist transfer supplement)

The frozen no-DFCI feature-set checklist with each item threshold rounded to a practical value
(`fcc.guideline_threshold`): below 1 to the nearest quarter, single digits to the nearest whole number, tens to the
nearest 5, hundreds to the nearest 10, thousands to the nearest 100, and so on. Item selection, unit weights and the
M-of-N rule are unchanged, so only the thresholds differ. The rounded checklists are scored on the same local frames,
bootstrap resamples and sequencing-clock frames as the exact ones and written to `checklist_grid_guidelines/`
(`transfer_checklist_patients.csv`, `transfer_checklist_comparison.csv`, plus `transfer_checklist_guideline_items.csv`
with the rounded and exact thresholds) for the figures in `manuscript_renders/supplement/generalized_federated_checklist_transfer/`.
The frozen probability calibrator was fit on the exact item counts, so only the item count and flag are used.


In [ ]:
guideline_bundles = {stem + fcc.GUIDELINE_SUFFIX: fcc.guideline_bundle(bundles[stem]) for stem in CHECKLIST_STEMS}
if guideline_bundles:
    guide_res = fet.evaluate_transfer(local, guideline_bundles, **TRANSFER_KW)
    guide_res_seq = fet.evaluate_transfer(local_seq, guideline_bundles, **{**TRANSFER_KW, "exclusion": "none"})
    guide_items = pd.concat([fcc.guideline_items(guideline_bundles[stem + fcc.GUIDELINE_SUFFIX], bundles[stem])
                             for stem in CHECKLIST_STEMS], ignore_index=True)
    fcc.write_guideline_grid(*fcc.checklist_grid(guide_res["scores"], guide_res_seq["scores"], clinical_seq,
                                                 n_bootstrap=N_BOOTSTRAP, seed=SEED), guide_items, OUTPUT_DIR,
                            full_accuracy=fcc.checklist_full_accuracy_grid(guide_res["scores"], n_bootstrap=N_BOOTSTRAP,
                                                                           seed=SEED))
    display(guide_items[guide_items["config"].eq("both")])
else:
    print("no checklist bundle; guideline checklist inputs are not written")
